# Phase 5 — Legal Statute Identification with the pre-trained **Encoder–Decoder**: the six experiments (hierarchical, multi-label)

Phase-1 gave us a T5-style legal **encoder–decoder** (RMSNorm, relative-position bias, gated FFN, 9+9 layers, 512-token encoder window).
Here it is the pre-trained backbone for **multi-label statute identification** on long legal documents. Data loading, label space, thresholding,
metrics, ablation harness and submission format follow the GQLA Phase-5 notebook, so results are directly comparable.

```
document → chunks of 511 BPE tokens + <eos>   (exactly the encoder-input format used in Phase-1 pre-training)
        → pre-trained ENCODER (bidirectional)  → token states
        → chunk vector = [ mean | max | <eos> state | decoder state ]      (any subset, an ablation)
                                   └─ decoder fed a single <s>, cross-attending the chunk, i.e. the model's
                                      "what comes next?" summary of the chunk
        → chunk-level BiGRU / BiLSTM / small BERT-style encoder             (cross-chunk context)
        → attention pooling → sigmoid per statute
```

**What differs from the GQLA notebook (and why)**

| | GQLA (decoder-only) | Encoder–Decoder (this notebook) |
|---|---|---|
| token-level BiRNN on top | needed – a causal model only sees the left context | **dropped** – the encoder is already bidirectional |
| chunk summary | `[mean \| max \| last-token]` | `[mean \| max \| <eos> \| decoder-state]` |
| chunk format | 512 tokens incl. BOS/EOS | **511 tokens + `<eos>`**, no BOS (matches Phase-1 encoder inputs) |
| extra ablation | – | does the **decoder** state add information? (`parts=...+dec`, `unfreeze_decoder`) |

Two modes share everything after the backbone:
* **`cached`** – backbone frozen and run **once**; per-chunk vectors stored in fp16; only the chunk-level encoder is trained (seconds/epoch → use for ablations).
* **`token`** – backbone run on the fly; `peft=dict(method=...)` adapts the encoder (PEFT, partial or full fine-tuning); the legacy switches `unfreeze_top_k` / `unfreeze_decoder` still work.

**Speed:** the encoder uses fused (SDPA) attention with the relative-position bias + padding mask folded into one additive bias, fp16/bf16 autocast, and
token-budget batching for extraction. Cell *6b* self-tests the fast path against the original attention before anything expensive runs.

**The six experiments of the Phase-5 brief (§11.2)** — all start from the Phase-1 checkpoint, no external model:

| # | experiment | technique | what is trained | cell |
|---|---|---|---|---|
| E1 | `e1_lorma_plus` | **LoRMA+** (multiplicative low-rank, ACL Findings 2025) — *compulsory* | `A`,`B` of `(α/r·BA + I)·W0` on `q`,`v` of the top 4 encoder layers | RUN |
| E2 | `e2_adalora` | **AdaLoRA** (SVD form, importance-based rank pruning) | `P`,`Λ`,`Q` on all 7 matrices of the top 4 layers, rank 12 → 6 | RUN |
| E3 | `e3_qlora` | **QLoRA** (frozen 4-bit NF4 encoder + LoRA) | LoRA `A`,`B` on the top 4 layers; all 9 layers stored in 4 bit | RUN |
| E4 | `e4_adapter` | **Bottleneck adapters** (Houlsby: after attention and after FFN) | adapters of the top 4 layers | RUN |
| E5 | `e5_partial_ft` | **Partial fine-tuning** | the 3 top encoder layers + final norm | RUN |
| E6 | `e6_kd_student` | **Knowledge distillation** | student = first 4 encoder layers (frozen) + small BiGRU, teacher = ensemble of E1–E5 | RUN |

Optional extras (commented in RUN): LoRMA-π, prefix tuning, prompt tuning, full fine-tuning, the student without a teacher.
Cell *4c* proves that every PEFT method starts as the identity of the pre-trained encoder.

> **All hyper-parameters in the RUN cell are placeholders** (4 epochs, ranks, learning rates, temperature …) chosen only so that the notebook runs end to end.
> Tuning them and interpreting the results is your job. Run **one experiment per cell first** and read the `s/batch -> ~N min per epoch` line before queueing the rest —
> token-mode experiments run the 9-layer encoder at every step. The brief also asks you to cover all four Phase-1 architectures and submit your best six overall.
> If you use an LLM for any of this, declare it as the course rules (§17.5) require.

**Run order:** top to bottom. Needs the Phase-1 checkpoint (`encoder_decoder_best.pt`), the Phase-1 tokenizer (`final_bpe_en.json`) and the Phase-1 notebook
(only to import `BPETokenizer` / `normalize_text` so they are byte-identical to training). Put your Hugging Face token in a Colab *secret* named `HF_TOKEN`.

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import os, json, math, time, random, copy, pickle, glob, shutil, gc
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

# ---- Hugging Face token: Colab secret or environment variable. Never hard-code it. ----
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
print("HF token available:", bool(HF_TOKEN))

# ---- paths (edit if yours differ) ------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/Phase1_Pretraining"
WORK_DIR   = os.path.join(DRIVE_ROOT, "phase1_encoder_decoder")
CKPT_DIR   = os.path.join(WORK_DIR, "checkpoints")
ckpt_path  = os.path.join(CKPT_DIR, "encoder_decoder_best.pt")          # Phase-1 best model
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(CKPT_DIR, "encoder_decoder_latest.pt")
tok_path   = os.path.join(WORK_DIR, "final_bpe_en.json")                # tokenizer used for Phase-1
p5_dir     = os.path.join(WORK_DIR, "phase5"); os.makedirs(p5_dir, exist_ok=True)

# the Phase-1 notebook, only used to import BPETokenizer + normalize_text (set it by hand if not found)
PHASE1_NB = None
for pat in ("/content/drive/MyDrive/Colab Notebooks/phase1_encoder_decoder*.ipynb",
            os.path.join(WORK_DIR, "*.ipynb"), "/content/phase1_encoder_decoder*.ipynb"):
    hits = sorted(glob.glob(pat))
    if hits:
        PHASE1_NB = hits[0]; break

for what, p in (("checkpoint", ckpt_path), ("tokenizer", tok_path)):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Phase-1 {what} not found: {p}  -> edit the paths above")
print("checkpoint:", ckpt_path)
print("phase-1 notebook:", PHASE1_NB)


def load_definitions_from_notebook(nb_path):
    """exec the cells that DEFINE BPETokenizer and normalize_text in the Phase-1 notebook, so the
    tokenisation / normalisation here is identical to the one the model was pre-trained with."""
    if nb_path is None:
        raise FileNotFoundError("Phase-1 notebook not found: set PHASE1_NB, or run its BPETokenizer and "
                                "normalize_text cells in this runtime first.")
    nb = json.load(open(nb_path, encoding="utf-8"))
    picked = []
    for marker in ("class BPETokenizer", "def normalize_text"):
        for i, c in enumerate(nb["cells"]):
            if c["cell_type"] == "code" and marker in "".join(c["source"]) and i not in picked:
                picked.append(i); break
        else:
            raise RuntimeError(f"no cell containing {marker!r} in {nb_path}")
    for i in sorted(picked):
        src = "".join(nb["cells"][i]["source"])
        src = "\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%")))
        exec(compile(src, f"<phase1 cell {i}>", "exec"), globals())
    print("loaded definitions from Phase-1 cells", sorted(picked))


if not all(n in globals() for n in ("BPETokenizer", "normalize_text")):
    load_definitions_from_notebook(PHASE1_NB)

tokenizer = BPETokenizer.from_json(tok_path)
PAD_ID, BOS_ID, EOS_ID = tokenizer.stoi["<pad>"], tokenizer.stoi["<s>"], tokenizer.stoi["</s>"]
print(f"tokenizer vocab: {len(tokenizer.itos):,} | PAD {PAD_ID}  BOS {BOS_ID}  EOS {EOS_ID}")

Mounted at /content/drive
device: cuda | Tesla T4
HF token available: True
checkpoint: /content/drive/MyDrive/Phase1_Pretraining/phase1_encoder_decoder/checkpoints/encoder_decoder_best.pt
phase-1 notebook: /content/drive/MyDrive/Colab Notebooks/phase1_encoder_decoder_bpe_pretraining_txl_data_pipeline-2.ipynb
loaded definitions from Phase-1 cells [7, 13]
tokenizer vocab: 32,768 | PAD 0  BOS 2  EOS 3


In [ ]:
# ============================================================
# 1. PRE-TRAINED ENCODER-DECODER  (architecture copied from Phase 1 so the state-dict loads strictly)
#
# Only two changes, both numerically equivalent to the original:
#   * Encoder.forward folds [relative-position bias + key-padding mask] into ONE additive bias, built once
#     per forward and shared by all layers, and attention uses F.scaled_dot_product_attention (fused,
#     no softmax matrix kept). FAST_ATTN["on"]=False restores the original manual attention.
#   * Encoder.forward(n_frozen=k): the first k layers run under no_grad (frozen part of a partly fine-tuned encoder).
# Added for the six Phase-5 experiments (all inactive -> numerically identical to the original unless a PEFT method turns them on):
#   * Encoder.forward(n_used=k): run only the first k layers (a truncated encoder, the distillation student); the output is then
#     normalised with a parameter-free RMS norm because final_ln was trained for the output of layer 9.
#   * Encoder.forward(grad_ckpt=True): gradient checkpointing of the layers that need gradients.
#   * EncoderLayer.adapter_attn / adapter_ff: optional bottleneck adapters after each sub-layer (Houlsby placement).
#   * MultiHeadAttention.prefix_k / prefix_v: optional learned key/value prefixes (prefix tuning, encoder self-attention only).
#   * the q/k/v/o and wi_0/wi_1/wo nn.Linear layers can be replaced by LoRMA / AdaLoRA / NF4 modules (cell 'PEFT for the encoder-decoder').
# The decoder is untouched (we only ever run it for ONE query token, so it is cheap).
# ============================================================
from torch.utils.checkpoint import checkpoint as grad_checkpoint

FAST_ATTN = {"on": True}


def rms_normalize(x, eps=1e-6):
    """RMSNorm without a learned weight (used on the output of an intermediate encoder layer)."""
    return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        var = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(var + self.eps)
        return self.weight * x


class RelativePositionBias(nn.Module):
    """T5-style relative position bias, bucketed on a log scale for far-apart positions."""

    def __init__(self, num_heads, num_buckets=32, max_distance=128, bidirectional=True):
        super().__init__()
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def _bucket(relative_position, bidirectional, num_buckets, max_distance):
        ret = 0
        n = -relative_position
        if bidirectional:
            num_buckets //= 2
            ret += (n < 0).to(torch.long) * num_buckets
            n = torch.abs(n)
        else:
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_if_large = max_exact + (
            torch.log(n.float() / max_exact + 1e-6)
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        val_if_large = torch.clamp(val_if_large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, val_if_large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        rel_pos = k_pos - q_pos
        bucket = self._bucket(rel_pos, self.bidirectional, self.num_buckets, self.max_distance)
        values = self.embedding(bucket)                    # q_len, k_len, num_heads
        return values.permute(2, 0, 1).unsqueeze(0)        # 1, num_heads, q_len, k_len


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.prefix_k = None             # prefix tuning: nn.Parameter [H, P, d_head] (set by apply_peft)
        self.prefix_v = None

    def forward(self, x_q, x_kv, mask=None, position_bias=None, attn_bias=None):
        B, Lq, D = x_q.shape
        Lk = x_kv.shape[1]
        q = self.q(x_q).view(B, Lq, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k(x_kv).view(B, Lk, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v(x_kv).view(B, Lk, self.num_heads, self.d_head).transpose(1, 2)

        if self.prefix_k is not None:        # P learned key/value vectors that every query may attend to (no position bias, never masked)
            P = self.prefix_k.size(1)
            k = torch.cat([self.prefix_k.unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype), k], dim=2)
            v = torch.cat([self.prefix_v.unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype), v], dim=2)
            if attn_bias is not None:
                attn_bias = F.pad(attn_bias, (P, 0))                      # zero bias on the prefix keys
            else:
                if position_bias is not None:
                    position_bias = F.pad(position_bias, (P, 0))
                if mask is not None:
                    mask = F.pad(mask, (P, 0), value=1)

        if attn_bias is not None:            # fused path: softmax(q k^T / sqrt(d_head) + attn_bias) v
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=attn_bias.to(q.dtype),
                dropout_p=self.dropout.p if self.training else 0.0)
        else:                                # original path
            scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
            if position_bias is not None:
                scores = scores + position_bias
            if mask is not None:
                scores = scores.masked_fill(mask == 0, float("-inf"))
            attn = self.dropout(torch.softmax(scores, dim=-1))
            out = torch.matmul(attn, v)
        return self.o(out.transpose(1, 2).contiguous().view(B, Lq, D))


class GatedFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.wi_0 = nn.Linear(d_model, d_ff, bias=False)
        self.wi_1 = nn.Linear(d_model, d_ff, bias=False)
        self.wo = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        h = self.act(self.wi_0(x)) * self.wi_1(x)
        return self.wo(self.dropout(h))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.ff = GatedFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)
        self.adapter_attn = None         # bottleneck adapters (set by apply_peft); None = original layer
        self.adapter_ff = None

    def forward(self, x, mask, position_bias, attn_bias=None):
        h = self.ln1(x)
        a = self.self_attn(h, h, mask, position_bias, attn_bias)
        if self.adapter_attn is not None:
            a = self.adapter_attn(a)
        x = x + self.dropout(a)
        h = self.ln2(x)
        f = self.ff(h)
        if self.adapter_ff is not None:
            f = self.adapter_ff(f)
        x = x + self.dropout(f)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln2 = RMSNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln3 = RMSNorm(d_model)
        self.ff = GatedFeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_mask, cross_mask, self_pos_bias):
        h = self.ln1(x)
        x = x + self.dropout(self.self_attn(h, h, self_mask, self_pos_bias))
        h = self.ln2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, cross_mask, None))
        h = self.ln3(x)
        x = x + self.dropout(self.ff(h))
        return x


class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.pos_bias = RelativePositionBias(num_heads, bidirectional=True)
        self.final_ln = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask, n_frozen=0, n_used=None, grad_ckpt=False):
        L = x.shape[1]
        pos_bias = self.pos_bias(L, L, x.device)                      # [1,H,L,L]
        mask = attention_mask[:, None, None, :]                       # [B,1,1,L]
        attn_bias = None
        if FAST_ATTN["on"]:
            with torch.no_grad():                                     # dtype the attention will run in (fp32 or autocast)
                dt = F.linear(x[:, :1, :1], x.new_zeros(1, 1)).dtype
            keep = attention_mask.bool()[:, None, None, :]
            attn_bias = pos_bias.to(dt).expand(x.size(0), -1, -1, -1).masked_fill(~keep, float("-inf"))   # [B,H,L,L], built once, shared by all layers
        x = self.dropout(x)
        n_run = len(self.layers) if n_used is None else n_used
        for i, layer in enumerate(self.layers[:n_run]):
            if i < n_frozen:                                          # frozen bottom layers: no graph, no memory
                with torch.no_grad():
                    x = layer(x, mask, pos_bias, attn_bias)
            elif grad_ckpt:                                           # layers with gradients: recompute activations in backward
                x = grad_checkpoint(layer, x, mask, pos_bias, attn_bias, use_reentrant=False)
            else:
                x = layer(x, mask, pos_bias, attn_bias)
        return self.final_ln(x) if n_used is None else rms_normalize(x)


class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.pos_bias = RelativePositionBias(num_heads, bidirectional=False)
        self.final_ln = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_attention_mask, cross_attention_mask):
        L = x.shape[1]
        causal = torch.tril(torch.ones(L, L, device=x.device, dtype=torch.bool))
        pad = self_attention_mask[:, None, None, :].bool()
        self_mask = (causal[None, None] & pad).float()
        cross_mask = cross_attention_mask[:, None, None, :]
        pos_bias = self.pos_bias(L, L, x.device)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, enc_out, self_mask, cross_mask, pos_bias)
        return self.final_ln(x)


class LegalSeq2SeqLM(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=16, d_ff=2048,
                 num_encoder_layers=9, num_decoder_layers=9, dropout=0.1, pad_id=0):
        super().__init__()
        self.shared_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_encoder_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_decoder_layers, dropout)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.shared_embed.weight  # tied embeddings

    def forward(self, input_ids, attention_mask, decoder_input_ids, decoder_attention_mask):
        enc_out = self.encoder(self.shared_embed(input_ids), attention_mask)
        dec_out = self.decoder(
            self.shared_embed(decoder_input_ids), enc_out, decoder_attention_mask, attention_mask
        )
        return self.lm_head(dec_out)


def load_pretrained_ed():
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    c = ck.get("model_config", {})
    m = LegalSeq2SeqLM(
        vocab_size=c.get("vocab_size", len(tokenizer.itos)), d_model=c.get("d_model", 512),
        num_heads=c.get("num_heads", 16), d_ff=c.get("d_ff", 2048),
        num_encoder_layers=c.get("num_encoder_layers", 9), num_decoder_layers=c.get("num_decoder_layers", 9),
        dropout=c.get("dropout", 0.1), pad_id=c.get("pad_id", PAD_ID))
    if m.shared_embed.num_embeddings != len(tokenizer.itos):
        raise ValueError(f"model vocab {m.shared_embed.num_embeddings} != tokenizer vocab {len(tokenizer.itos)}: "
                         f"wrong tokenizer file for this checkpoint")
    m.load_state_dict(ck["model_state_dict"])            # strict: fails loudly on any mismatch
    print(f"loaded Phase-1 checkpoint | step {ck.get('global_step')} | best val token-loss {ck.get('best_val_loss')}")
    return m.eval()


import hashlib
CKPT_SIG = hashlib.md5(f"{os.path.basename(ckpt_path)}_{os.path.getsize(ckpt_path)}".encode()).hexdigest()[:8]   # keys the feature cache
ED = load_pretrained_ed().to(device)
D_ED = ED.shared_embed.embedding_dim
print(f"Encoder-decoder loaded: {sum(p.numel() for p in ED.parameters())/1e6:.1f}M params | d_model {D_ED} | "
      f"{len(ED.encoder.layers)} enc + {len(ED.decoder.layers)} dec layers")

loaded Phase-1 checkpoint | step 11200 | best val token-loss 4.871949663513102
Encoder-decoder loaded: 101.7M params | d_model 512 | 9 enc + 9 dec layers


In [ ]:
# ============================================================
# 2. LOAD PHASE-5 (LSI) DATA   (same reading logic as the GQLA notebook)
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"


def find_repo_file(basename):
    """Locate e.g. train.json inside the Phase-5 folder without hard-coding its exact name."""
    files = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)
    hits = [f for f in files if f.endswith("/" + basename) and ("lsi" in f.lower() or "phase_5" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 5, got {hits}. "
                           f"Set the path by hand. All files: {files[:40]}")
    return hits[0]


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):                       # {id: {text, labels}}  (Phase-1 style)
        raw = [{**v, "id": k} for k, v in raw.items()]
    recs = []
    for r in raw:                                   # [{id, text, labels}]  (Phase-5 spec)
        recs.append(dict(id=r["id"], text=r["text"], labels=[int(l) for l in r.get("labels", [])]))
    return recs


train_recs, dev_recs, test_recs = read_split("train.json"), read_split("dev.json"), read_split("test.json")
print(len(train_recs), len(dev_recs), len(test_recs), "train/dev/test documents")

# ---- label space: compact index <-> statute id ---------------------------------------------
all_labels = sorted({l for r in train_recs + dev_recs for l in r["labels"]})
lab2idx = {l: i for i, l in enumerate(all_labels)}
idx2lab = {i: l for l, i in lab2idx.items()}
L = len(all_labels)


def multi_hot(recs):
    y = np.zeros((len(recs), L), dtype=np.float32)
    for i, r in enumerate(recs):
        for l in r["labels"]:
            y[i, lab2idx[l]] = 1.0
    return y


y_train, y_dev = multi_hot(train_recs), multi_hot(dev_recs)
freq = y_train.sum(0)
print(f"{L} statutes | labels/doc: train {y_train.sum(1).mean():.2f}")
print(f"train docs per statute: min {freq.min():.0f}  median {np.median(freq):.0f}  max {freq.max():.0f}")
print(f"statutes with <=5 train docs: {(freq <= 5).sum()}   in dev but never in train: "
      f"{((y_dev.sum(0) > 0) & (freq == 0)).sum()}")


# ---- tokenise once (full documents, NO special tokens), cache to Drive ----------------------------
# Phase-1 encoder inputs were  bpe(text)[:511] + [</s>]  with no <s>, so special tokens are added per CHUNK later.
def encode_docs(recs, tag, bs=2000, workers=None):
    workers = workers or min(4, os.cpu_count() or 1)
    path = os.path.join(p5_dir, f"tok_ns_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(recs), bs):
        texts = [normalize_text(r["text"]) for r in recs[i:i + bs]]        # same normalisation as Phase 1
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(texts, add_special_tokens=False, num_workers=workers, chunk_size=32)]
        print(f"  tokenised {min(i + bs, len(recs)):,}/{len(recs):,} ({tag})")
    pickle.dump(out, open(path, "wb"))
    return out


enc_train, enc_dev, enc_test = (encode_docs(train_recs, "train"), encode_docs(dev_recs, "dev"),
                                encode_docs(test_recs, "test"))
lens = np.array([len(x) for x in enc_train])
print("train tokens/doc  median %d  p90 %d  p99 %d  max %d" % tuple(np.percentile(lens, [50, 90, 99, 100]).astype(int)))
print("-> 511-token chunks/doc  median %.1f  p90 %.1f" % tuple(np.percentile(np.ceil(lens / 511), [50, 90])))

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

42750 10181 13019 train/dev/test documents
100 statutes | labels/doc: train 3.70
train docs per statute: min 272  median 679  max 11730
statutes with <=5 train docs: 0   in dev but never in train: 0
train tokens/doc  median 637  p90 4148  p99 12289  max 140139
-> 511-token chunks/doc  median 2.0  p90 9.0


In [ ]:
# ============================================================
# Pure-numpy utilities: chunking, metrics, thresholds  (identical to the GQLA notebook, except chunk_ids
# is given the BODY length 511: one <eos> is appended per chunk later, so every window is <= 512 tokens)
# ============================================================
import numpy as np


# ---------------- long-document chunking ----------------
def chunk_ids(ids, body_len=511, stride=511, min_tail=16):
    """Split one tokenised document into windows of <= body_len tokens (an <eos> is appended later).
    stride == body_len -> non-overlapping windows; stride < body_len -> overlapping windows."""
    if len(ids) <= body_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + body_len])
        if s + body_len >= len(ids):
            break
        s += stride
    # a tiny trailing fragment carries almost no information
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="head_tail"):
    """Long-document representation strategy (one of the ablations)."""
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "tail":
        return chunks[-max_chunks:]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(f"unknown strategy {strategy!r}")


# ---------------- metrics ----------------
def _counts(y, pred):
    tp = (y * pred).sum(0)
    fp = ((1 - y) * pred).sum(0)
    fn = (y * (1 - pred)).sum(0)
    return tp, fp, fn


def f1_per_class(y, pred):
    tp, fp, fn = _counts(y, pred)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0), y.sum(0)


def compute_metrics(y, pred):
    """macro_f1      : mean F1 over statutes that occur in `y` (primary metric)
       macro_f1_all  : mean F1 over every column, absent statutes count as 0
       micro_f1, subset_acc : the two secondary metrics"""
    f1, sup = f1_per_class(y, pred)
    tp, fp, fn = _counts(y, pred)
    return dict(
        macro_f1=float(f1[sup > 0].mean()) if (sup > 0).any() else 0.0,
        macro_f1_all=float(f1.mean()),
        micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
        subset_acc=float((y == pred).all(1).mean()),
    )


# ---------------- thresholding ----------------
THR_GRID = np.round(np.arange(0.05, 0.951, 0.025), 4)


def predict_labels(probs, thr, force_top1=True):
    """thr: scalar or per-class array [L]. Every document has >=1 statute, so if
    nothing clears the threshold we fall back to the arg-max statute."""
    pred = (probs >= thr).astype(np.float32)
    if force_top1:
        empty = np.where(pred.sum(1) == 0)[0]
        if len(empty):
            pred[empty, probs[empty].argmax(1)] = 1.0
    return pred


def tune_global_threshold(probs, y, grid=THR_GRID, force_top1=True):
    scores = [compute_metrics(y, predict_labels(probs, t, force_top1))["macro_f1"] for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])


def tune_per_class_threshold(probs, y, global_thr, grid=THR_GRID, min_support=3, shrink=0.5):
    """Per-statute thresholds. Statutes with < min_support positives in the tuning
    set keep the global threshold (too little evidence to tune); the rest are
    shrunk towards it:  thr_c = shrink*best_c + (1-shrink)*global."""
    n, L = probs.shape
    thr = np.full(L, global_thr, dtype=np.float32)
    sup = y.sum(0)
    for c in range(L):
        if sup[c] < min_support:
            continue
        P = probs[:, c:c + 1] >= grid[None, :]
        yc = y[:, c:c + 1] > 0
        tp = (P & yc).sum(0)
        fp = (P & ~yc).sum(0)
        fn = (~P & yc).sum(0)
        f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        best = grid[np.flatnonzero(f1 == f1.max())]           # ties -> closest to global
        best = best[np.argmin(np.abs(best - global_thr))]
        thr[c] = shrink * best + (1 - shrink) * global_thr
    return thr


def honest_threshold_eval(probs, y, min_support=2, shrink=0.5, seed=0):
    """Tuning thresholds on dev and scoring on the same dev is optimistic. This
    tunes on one random half of dev and scores on the other half (and swaps)."""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(y))
    A, B = perm[: len(y) // 2], perm[len(y) // 2:]
    res = {"fixed_0.5": [], "global": [], "per_class": []}
    for tr, te in ((A, B), (B, A)):
        g, _ = tune_global_threshold(probs[tr], y[tr])
        pc = tune_per_class_threshold(probs[tr], y[tr], g, min_support=min_support, shrink=shrink)
        res["fixed_0.5"].append(compute_metrics(y[te], predict_labels(probs[te], 0.5))["macro_f1"])
        res["global"].append(compute_metrics(y[te], predict_labels(probs[te], g))["macro_f1"])
        res["per_class"].append(compute_metrics(y[te], predict_labels(probs[te], pc))["macro_f1"])
    return {k: float(np.mean(v)) for k, v in res.items()}


def rare_report(y_dev, pred, y_train, edges=((0, 0), (1, 5), (6, 20), (21, 100), (101, 10**9))):
    """Mean F1 on dev, bucketed by how many TRAIN documents carry the statute."""
    f1, sup = f1_per_class(y_dev, pred)
    tf = y_train.sum(0)
    rows = []
    for lo, hi in edges:
        m = (tf >= lo) & (tf <= hi) & (sup > 0)
        rows.append((f"{lo}-{hi if hi < 10**9 else 'inf'}", int(m.sum()),
                     float(f1[m].mean()) if m.any() else float("nan")))
    return rows

In [ ]:
# ============================================================
# PEFT FOR THE ENCODER-DECODER  -  the adaptation techniques of the Phase-5 brief, applied to the Phase-1 T5-style encoder
#
#   method        technique (reference)                              what is trained
#   ------------  -------------------------------------------------  ------------------------------------------
#   lorma_plus    LoRMA+  (Bihany, Patel, Modi, ACL-Findings 2025)    B, A of  (a/r*BA + I) W0   [multiplicative]
#   lorma_pi      LoRMA-pi (same paper, permutation inflation)        B, A of  I_pi(BA) W0
#   adalora       AdaLoRA (Zhang et al., 2023)                        P, Lambda, Q (SVD form) + rank pruning
#   qlora         QLoRA   (Dettmers et al., 2023)                     LoRA on top of a frozen 4-bit NF4 encoder
#   adapter       Bottleneck adapters (Houlsby et al., 2019)          down/up projections after attention AND after the FFN
#   prefix        Prefix tuning (Li & Liang, 2021)                    P learned key/value vectors per encoder layer
#   prompt        Prompt tuning (Lester et al., 2021)                 P soft tokens prepended to every chunk
#   partial       Partial / layer-wise fine-tuning                    top-k encoder layers (+ final norm)
#   full          Full fine-tuning                                    every encoder parameter incl. the embeddings
#
# The encoder's matrices are q, k, v, o (self-attention, 512x512) and wi_0, wi_1 (512->2048), wo (2048->512) in every layer.
# LoRMA adapts q and v as in the paper; AdaLoRA and QLoRA adapt all seven. By default the new modules live in the TOP
# `top_layers` encoder layers: the layers underneath run under no_grad (no activation memory), which is what makes
# token-mode training feasible on a T4. The decoder is never adapted (it only reads one <s> token per chunk).
#
# Use:  run_experiment(name, mode="token", peft=dict(method="lorma_plus", r=8), ...)   (see the RUN cell)
# ============================================================
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# NormalFloat-4 code book (Dettmers et al., 2023): quantiles of N(0,1) normalised to [-1, 1], with an exact zero.
NF4_TABLE = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
             -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
             0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
             0.7229568362236023, 1.0]


ALL_LINEARS = ("q", "k", "v", "o", "wi_0", "wi_1", "wo")

PEFT_DEFAULTS = {
    "lorma_plus": dict(r=8, alpha=16, top_layers=4, targets=("q", "v")),
    # I_pi materialises a (d_out x d_out) matrix per module -> square q/v (512x512) by default
    "lorma_pi":   dict(r=8, top_layers=4, targets=("q", "v")),
    "adalora":    dict(init_r=12, target_r=6, alpha=32, top_layers=4, targets=ALL_LINEARS,
                       t_init=0.1, t_final=0.2, delta_t=10, beta=0.85, orth=0.1),
    "qlora":      dict(r=16, alpha=32, top_layers=4, targets=ALL_LINEARS, quant_all_layers=True, blocksize=64),
    "adapter":    dict(bottleneck=64, top_layers=4),
    "prefix":     dict(n_tokens=16, top_layers=4),
    "prompt":     dict(n_tokens=20),
    "partial":    dict(k=3),
    "full":       dict(decoder=False),
}


def _owner(layer, name):
    """the sub-module of an EncoderLayer that owns the nn.Linear called `name`"""
    return layer.self_attn if name in ("q", "k", "v", "o") else layer.ff


class PEFTHandle:
    """Everything train_model needs to know about the chosen method."""

    def __init__(self, method, spec):
        self.method, self.spec = method, spec
        self.params = []              # NEW trainable parameters created by the method (they get cfg.lr_peft)
        self.mods = []                # injected modules
        self.allocator = None         # AdaLoRA rank allocator

    def param_ids(self):
        return {id(p) for p in self.params}

    def setup(self, total_steps):
        if self.allocator is not None:
            self.allocator.setup(total_steps)

    def extra_loss(self):
        return self.allocator.orth_loss() if self.allocator is not None else 0.0

    def after_backward(self, step):                 # gradients are un-scaled here, before clipping
        if self.allocator is not None:
            self.allocator.update_importance()

    def after_step(self, step):                     # optimizer step done
        if self.allocator is not None:
            self.allocator.after_step(step)

    def describe(self):
        n = sum(p.numel() for p in self.params)
        extra = ""
        if self.allocator is not None:
            r = self.allocator.effective_ranks()
            extra = f" | AdaLoRA ranks now: min {min(r)} / mean {sum(r)/len(r):.1f} / max {max(r)}"
        return f"PEFT '{self.method}': {n/1e6:.3f}M new parameters in {len(self.mods)} modules{extra}"


# ------------------------------------------------------------------ LoRMA
class LoRMALinear(nn.Module):
    """Low-Rank MULTIPLICATIVE Adaptation:  h = I(BA) W0 x     (W0: frozen [d x k],  B: [d x r],  A: [r x d])

    'plus' (LoRMA+): I+(BA) = alpha/r * BA + I_d.   B = 0, A random  ->  identity at init.
                     Evaluated in the cheap order  y + alpha/r * B (A y)  with y = W0 x  (no d x d matrix is built).
    'pi'   (LoRMA-pi): I_pi(BA) rotates row i of BA cyclically by i, which lifts a rank-r matrix to (almost) full rank.
                     Init B[:,0]=1, rest random; A[0,0]=1, rest 0  ->  I_pi(BA) = I_d.   (no alpha in this variant)"""

    def __init__(self, base, r=8, alpha=16, variant="plus"):
        super().__init__()
        assert base.bias is None, "the encoder-decoder linears have no bias"
        d = base.out_features
        self.weight = base.weight                                   # frozen, shares storage with the original layer
        self.variant, self.r, self.scale = variant, r, alpha / r
        self.A, self.B = nn.Parameter(torch.empty(r, d)), nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            self.A.data[0, 0] = 1.0
            nn.init.normal_(self.B, std=0.02)
            self.B.data[:, 0] = 1.0
            idx = (torch.arange(d)[None, :] - torch.arange(d)[:, None]) % d     # M_rot[i, j] = M[i, (j - i) mod d]
            self.register_buffer("rot_idx", idx, persistent=False)

    def forward(self, x):
        y = F.linear(x, self.weight)                                # W0 x
        if self.variant == "plus":
            return y + self.scale * F.linear(F.linear(y, self.A), self.B)
        M = torch.gather(self.B @ self.A, 1, self.rot_idx)          # I_pi(BA)  [d x d]
        return F.linear(y, M.to(y.dtype))


# ------------------------------------------------------------------ AdaLoRA
class AdaLoRALinear(nn.Module):
    """W0 x + alpha/r_init * P diag(E) Q x   (SVD-style update; E = singular values, pruned by AdaLoRAAllocator)."""

    def __init__(self, base, r=12, alpha=32):
        super().__init__()
        d, k = base.out_features, base.in_features
        self.weight, self.scale = base.weight, alpha / r
        self.P = nn.Parameter(torch.randn(d, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, k) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))

    def forward(self, x):
        y = F.linear(x, self.weight)
        return y + self.scale * F.linear(F.linear(x, self.Q) * self.E, self.P)


class AdaLoRAAllocator:
    """Importance-based rank allocation (Zhang et al., 2023, Sec. 3.2-3.3).
       sensitivity  I(w) = |w * dL/dw|, smoothed  I_bar <- b I_bar + (1-b) I,  U <- b U + (1-b)|I - I_bar|,  s(w) = I_bar * U
       triplet i    S_i = s(E_i) + mean_j s(P_ji) + mean_j s(Q_ij)
       every delta_t steps the globally lowest-scoring triplets are zeroed so that only b(t) remain, with the cubic
       schedule  b(t) = b_T + (b_0 - b_T) (1 - (t-t_i)/(T-t_i-t_f))^3.  In the last t_f steps the pattern is frozen.
       Orthogonality regulariser  orth * mean_modules( ||P^T P - I||_F + ||Q Q^T - I||_F )."""

    def __init__(self, mods, target_r, t_init, t_final, delta_t, beta, orth):
        self.mods, self.delta_t, self.beta, self.orth = mods, delta_t, beta, orth
        self.t_init_frac, self.t_final_frac = t_init, t_final
        self.b0 = sum(m.E.numel() for m in mods)
        self.bT = int(target_r * len(mods))
        self.ipt, self.unc, self.keep, self.total = {}, {}, None, None

    def setup(self, total_steps):
        self.total = total_steps
        self.ti, self.tf = int(self.t_init_frac * total_steps), int(self.t_final_frac * total_steps)
        self.keep = None

    def orth_loss(self):
        loss = 0.0
        for m in self.mods:
            I = torch.eye(m.P.size(1), device=m.P.device)
            loss = loss + torch.norm(m.P.t() @ m.P - I) + torch.norm(m.Q @ m.Q.t() - I)
        return self.orth * loss / len(self.mods)

    @torch.no_grad()
    def update_importance(self):
        b = self.beta
        for m in self.mods:
            for p in (m.P, m.E, m.Q):
                if p.grad is None:
                    continue
                s = torch.nan_to_num((p * p.grad).abs(), nan=0.0, posinf=0.0, neginf=0.0)
                k = id(p)
                if k not in self.ipt:
                    self.ipt[k], self.unc[k] = torch.zeros_like(p), torch.zeros_like(p)
                self.ipt[k].mul_(b).add_(s, alpha=1 - b)
                self.unc[k].mul_(b).add_((s - self.ipt[k]).abs(), alpha=1 - b)

    @torch.no_grad()
    def _scores(self):
        out = []
        for m in self.mods:
            sE = self.ipt.get(id(m.E), torch.zeros_like(m.E)) * self.unc.get(id(m.E), torch.zeros_like(m.E))
            sP = self.ipt.get(id(m.P), torch.zeros_like(m.P)) * self.unc.get(id(m.P), torch.zeros_like(m.P))
            sQ = self.ipt.get(id(m.Q), torch.zeros_like(m.Q)) * self.unc.get(id(m.Q), torch.zeros_like(m.Q))
            out.append(sE + sP.mean(0) + sQ.mean(1))                  # [r] per module
        return out

    @torch.no_grad()
    def _prune_to(self, budget):
        scores = self._scores()
        allv = torch.cat(scores)
        if budget >= allv.numel():
            return [torch.ones_like(s, dtype=torch.bool) for s in scores]
        thr = torch.topk(allv, budget).values[-1]
        keep = [s >= thr for s in scores]
        for m, kp in zip(self.mods, keep):
            m.E.data.mul_(kp.to(m.E.dtype))                          # pruned singular values -> 0 (can be revived by gradient)
        return keep

    def budget_at(self, step):
        mul = 1 - (step - self.ti) / max(1, self.total - self.tf - self.ti)
        return int(self.bT + (self.b0 - self.bT) * mul ** 3)

    @torch.no_grad()
    def after_step(self, step):
        if self.total is None or step <= self.ti:
            return                                                    # warm-up at full rank
        if step > self.total - self.tf:                              # final phase: fixed rank pattern
            if self.keep is None:
                self.keep = self._prune_to(self.bT)
            for m, kp in zip(self.mods, self.keep):
                m.E.data.mul_(kp.to(m.E.dtype))
        elif step % self.delta_t == 0:
            self._prune_to(self.budget_at(step))

    def effective_ranks(self):
        return [int((m.E != 0).sum()) for m in self.mods]


# ------------------------------------------------------------------ QLoRA (NF4 base + LoRA)
class NF4Linear(nn.Module):
    """Frozen weight stored as 4-bit NormalFloat codes (two per byte) + one fp16 absmax per `blocksize` weights,
       de-quantised on the fly in the forward pass; optional LoRA update  W x + alpha/r * B A x  (A, B in 16/32 bit).
       Simplifications w.r.t. the QLoRA paper: no double quantisation of the absmax constants, no paged optimiser."""

    def __init__(self, base, r=0, alpha=16, blocksize=64):
        super().__init__()
        w = base.weight.data.detach().float()
        self.out_features, self.in_features, self.blocksize, self.r = base.out_features, base.in_features, blocksize, r
        flat = w.reshape(-1)
        n = flat.numel()
        assert n % blocksize == 0 and (n // 2) * 2 == n
        blk = flat.view(-1, blocksize)
        absmax = blk.abs().amax(1).clamp_min(1e-8)
        normed = (blk / absmax[:, None]).reshape(-1)
        table = torch.tensor(NF4_TABLE, device=w.device)
        codes = torch.empty(n, dtype=torch.uint8, device=w.device)
        for s in range(0, n, 1 << 20):                                # nearest code-book entry, in slices
            codes[s:s + (1 << 20)] = (normed[s:s + (1 << 20), None] - table[None, :]).abs().argmin(1).to(torch.uint8)
        self.register_buffer("qweight", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("table", table, persistent=False)
        self.n = n
        if r > 0:
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, self.in_features))
            self.B = nn.Parameter(torch.zeros(self.out_features, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def dequantize(self, dtype):
        codes = torch.stack([(self.qweight >> 4).long(), (self.qweight & 15).long()], 1).reshape(-1)
        w = self.table[codes].view(-1, self.blocksize) * self.absmax.float()[:, None]
        return w.reshape(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        y = F.linear(x, self.dequantize(x.dtype))
        if self.r > 0:
            y = y + self.scale * F.linear(F.linear(x, self.A), self.B)
        return y


# ------------------------------------------------------------------ bottleneck adapter
class BottleneckAdapter(nn.Module):
    """y + W_up gelu(W_down y), W_up = 0 at init (identity). Houlsby placement: one adapter after the attention sub-layer and
       one after the FFN sub-layer of every adapted encoder layer (both applied before the residual connection)."""

    def __init__(self, d, bottleneck):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, y):
        return y + self.up(F.gelu(self.down(y)))


# ------------------------------------------------------------------ injection
def apply_peft(bb, spec):
    """Inject the method described by spec = dict(method=..., **overrides) into backbone `bb` (an EDBackbone that already
       froze all pre-trained weights). Returns a PEFTHandle. Call on a deep copy of the encoder-decoder."""
    spec = dict(spec)
    method = spec.pop("method")
    c = {**PEFT_DEFAULTS[method], **spec}
    h = PEFTHandle(method, c)
    enc = bb.ed.encoder
    n = bb.n_used
    first = 0 if c.get("top_layers") is None else max(0, n - c["top_layers"])

    if method == "partial":
        bb.unfreeze_layers(max(0, n - c["k"]))
    elif method == "full":
        bb.unfreeze_layers(0)
        bb.train_embed = True
        bb.ed.shared_embed.weight.requires_grad = True
        if c["decoder"]:
            bb.unfreeze_dec = True
            for p in bb.ed.decoder.parameters():
                p.requires_grad = True
    elif method == "prompt":
        emb = bb.ed.shared_embed.weight
        ids = torch.randint(5, emb.size(0), (c["n_tokens"],))        # init from random vocabulary rows (Lester et al.)
        bb.soft_prompt = nn.Parameter(emb.data[ids].clone().float())
        bb.first_grad = 0                                            # the prompt sits below every layer
        h.params.append(bb.soft_prompt)
    elif method == "prefix":
        for i in range(first, n):
            att = enc.layers[i].self_attn
            att.prefix_k = nn.Parameter(torch.randn(att.num_heads, c["n_tokens"], att.d_head, device=bb.device()) * 0.02)
            att.prefix_v = nn.Parameter(torch.randn(att.num_heads, c["n_tokens"], att.d_head, device=bb.device()) * 0.02)
            h.params += [att.prefix_k, att.prefix_v]
            h.mods.append(att)
        bb.first_grad = min(bb.first_grad, first)
    elif method == "adapter":
        for i in range(first, n):
            layer = enc.layers[i]
            layer.adapter_attn = BottleneckAdapter(bb.d_model, c["bottleneck"]).to(bb.device())
            layer.adapter_ff = BottleneckAdapter(bb.d_model, c["bottleneck"]).to(bb.device())
            for ad in (layer.adapter_attn, layer.adapter_ff):
                h.mods.append(ad)
                h.params += list(ad.parameters())
        bb.first_grad = min(bb.first_grad, first)
    else:                                                            # methods that replace q/k/v/o/wi_0/wi_1/wo
        quant_from = 0 if (method == "qlora" and c["quant_all_layers"]) else first
        for i in range(quant_from, n):
            layer = enc.layers[i]
            for name in c["targets"]:
                owner = _owner(layer, name)
                base = getattr(owner, name)
                adapted = i >= first
                if method in ("lorma_plus", "lorma_pi"):
                    new = LoRMALinear(base, c["r"], c.get("alpha", 16), "plus" if method == "lorma_plus" else "pi")
                    h.params += [new.A, new.B]
                elif method == "adalora":
                    new = AdaLoRALinear(base, c["init_r"], c["alpha"])
                    h.params += [new.P, new.Q, new.E]
                elif method == "qlora":
                    new = NF4Linear(base, c["r"] if adapted else 0, c["alpha"], c["blocksize"])
                    if adapted:
                        h.params += [new.A, new.B]
                else:
                    raise ValueError(method)
                new = new.to(base.weight.device)
                setattr(owner, name, new)
                if method != "qlora" or adapted:
                    h.mods.append(new)
        bb.first_grad = min(bb.first_grad, first)
        if method == "adalora":
            h.allocator = AdaLoRAAllocator(h.mods, c["target_r"], c["t_init"], c["t_final"],
                                           c["delta_t"], c["beta"], c["orth"])
    return h


# ------------------------------------------------------------------ size / cost accounting
def ed_prefix_footprint(ed, n_used, with_decoder=False):
    """(#params, bytes) of the part of the encoder-decoder that a deployed pipeline needs: embedding + the first n_used encoder
       layers (+ final norm if all are used) + positional-bias table + their PEFT modules, 4-bit buffers included.
       The decoder is counted only if the chunk vector uses its state ('dec' in parts). The tied lm_head is never needed."""
    n_total = len(ed.encoder.layers)
    params = nbytes = 0
    for name, p in ed.named_parameters():
        mt = re.match(r"encoder\.layers\.(\d+)\.", name)
        if mt and int(mt.group(1)) >= n_used:
            continue
        if name.startswith("decoder.") and not with_decoder:
            continue
        if name.startswith("encoder.final_ln") and n_used < n_total:
            continue
        params += p.numel(); nbytes += p.numel() * p.element_size()
    for name, m in ed.named_modules():
        mt = re.match(r"encoder\.layers\.(\d+)\.", name)
        if isinstance(m, NF4Linear) and not (mt and int(mt.group(1)) >= n_used):
            params += m.n; nbytes += m.qweight.numel() + m.absmax.numel() * 2
    return params, nbytes


def model_footprint(model, mode, n_used_cached=None, parts=None):
    """Deployable pipeline = encoder-decoder prefix that is used + everything trained on top of it."""
    if mode == "token":
        bb = model.backbone
        p0, b0 = ed_prefix_footprint(bb.ed, bb.n_used, "dec" in model.parts)
        head = [(n, p) for n, p in model.named_parameters() if not n.startswith("backbone.")]
        extra = [bb.soft_prompt] if bb.soft_prompt is not None else []
    else:
        p0, b0 = ed_prefix_footprint(ED, n_used_cached, "dec" in parts)
        head, extra = list(model.named_parameters()), []
    pn = sum(p.numel() for _, p in head) + sum(p.numel() for p in extra)
    bn = sum(p.numel() * p.element_size() for _, p in head) + sum(p.numel() * p.element_size() for p in extra)
    return (p0 + pn) / 1e6, (b0 + bn) / 1e6


In [ ]:
# ============================================================
# 3. MODEL: pre-trained ENCODER(-DECODER)  ->  chunk vector  ->  chunk-level BiGRU/BiLSTM/BERT-style  ->  attention pooling
#
# The encoder is bidirectional, so (unlike the GQLA decoder) no token-level BiRNN is needed: every token state already
# sees the whole 512-token chunk. The decoder adds a second view of the chunk: fed one <s> token and cross-attending the
# encoder states, its output is the model's summary of "what continues this text".
# ============================================================
from contextlib import nullcontext

PART_NAMES = ("mean", "max", "eos", "dec")          # fixed column order of the cached features


def _pack_run(rnn, x, lengths):
    """Run an RNN over right-padded x with packing so the backward direction starts at the real end."""
    packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
    out, _ = rnn(packed)
    out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
    return out


def _make_rnn(kind, d_in, hid, layers, dropout):
    cls = {"gru": nn.GRU, "lstm": nn.LSTM}[kind]
    return cls(d_in, hid, layers, batch_first=True, bidirectional=True, dropout=dropout if layers > 1 else 0.0)


class EDBackbone(nn.Module):
    """Wraps the pre-trained LegalSeq2SeqLM; returns encoder states (and optionally the decoder's <s>-step state).
       use_layer      : -1 = all encoder layers + final norm; k>=0 = output of encoder layer k (parameter-free RMS norm), i.e. a
                        TRUNCATED encoder of k+1 layers (the distillation student). The decoder then reads these shallower states.
       unfreeze_top_k : 0 = encoder fully frozen; k>0 = fine-tune the top k USED encoder layers (+ final norm if use_layer=-1)
       unfreeze_decoder: also fine-tune the decoder
       PEFT methods (cell 'PEFT for the encoder-decoder') inject trainable modules into the used layers and lower `first_grad`,
       the first encoder layer that must run with autograd. Layers below it run under no_grad (no activation memory); layers from
       it upwards are gradient-checkpointed. The whole model stays in eval mode (no dropout) even when the wrapper is in train mode."""

    def __init__(self, ed, unfreeze_top_k=0, unfreeze_decoder=False, use_layer=-1, grad_ckpt=True):
        super().__init__()
        self.ed, self.k, self.unfreeze_dec = ed, unfreeze_top_k, unfreeze_decoder
        self.use_layer, self.grad_ckpt = use_layer, grad_ckpt
        self.d_model = ed.shared_embed.embedding_dim
        n = len(ed.encoder.layers)
        self.n_used = n if use_layer == -1 else use_layer + 1            # encoder layers that contribute to the output
        for p in ed.parameters():
            p.requires_grad = False
        self.first_grad = self.n_used                                    # = nothing trainable inside the encoder
        self.soft_prompt, self.train_embed = None, False
        if unfreeze_top_k > 0:
            self.unfreeze_layers(max(0, self.n_used - unfreeze_top_k))
        if unfreeze_decoder:
            for p in ed.decoder.parameters():
                p.requires_grad = True

    def device(self):
        return self.ed.shared_embed.weight.device

    def unfreeze_layers(self, first):
        for layer in self.ed.encoder.layers[first:self.n_used]:
            for p in layer.parameters():
                p.requires_grad = True
        if self.use_layer == -1:
            for p in self.ed.encoder.final_ln.parameters():
                p.requires_grad = True
        self.first_grad = min(self.first_grad, first)

    def train(self, mode=True):
        super().train(mode)
        self.ed.eval()
        return self

    def forward(self, ids, tok_mask, need_dec=False):       # ids [M,T] long, tok_mask [M,T] bool
        ed = self.ed
        am = tok_mask.long()
        with torch.set_grad_enabled(self.train_embed and torch.is_grad_enabled()):
            x = ed.shared_embed(ids)
        P, am_full = 0, am
        if self.soft_prompt is not None:                      # prompt tuning: learned vectors prepended to every chunk
            P = self.soft_prompt.size(0)
            x = torch.cat([self.soft_prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype), x], dim=1)
            am_full = torch.cat([torch.ones(am.size(0), P, dtype=am.dtype, device=am.device), am], dim=1)
        enc = ed.encoder(x, am_full, n_frozen=min(self.first_grad, self.n_used),
                         n_used=None if self.use_layer == -1 else self.n_used,
                         grad_ckpt=self.grad_ckpt and self.training and torch.is_grad_enabled())
        if P:
            enc = enc[:, P:]                                  # the prompt positions only act through the encoder's attention
        dec = None
        if need_dec:
            bos = torch.full((ids.size(0), 1), BOS_ID, dtype=torch.long, device=ids.device)
            ones = torch.ones_like(bos)
            needs_grad = (self.first_grad < self.n_used or self.unfreeze_dec or self.soft_prompt is not None
                          or self.train_embed)
            with nullcontext() if needs_grad else torch.no_grad():          # gradients must reach the trainable part
                dec = ed.decoder(ed.shared_embed(bos), enc, ones, am)[:, 0]
        return enc, dec


def pool_parts(enc, tok_mask, dec=None):
    """Chunk summaries: mean / max over real tokens, the <eos> state (last real token = has attended to the whole
    chunk), and the decoder state."""
    enc = enc.float()
    m = tok_mask.unsqueeze(-1)
    ln = tok_mask.sum(1, keepdim=True)
    out = {"mean": (enc * m).sum(1) / ln,
           "max": enc.masked_fill(~m, -1e4).max(1).values,
           "eos": enc[torch.arange(enc.size(0), device=enc.device), ln.squeeze(1) - 1]}
    if dec is not None:
        out["dec"] = dec.float()
    return out


def parse_parts(s):
    parts = tuple(p for p in PART_NAMES if p in s.split("+"))        # canonical order
    bad = set(s.split("+")) - set(PART_NAMES)
    if bad or not parts:
        raise ValueError(f"parts={s!r}: use a '+'-joined subset of {PART_NAMES}")
    return parts


def chunk_vectors(backbone, ids, tok_mask, parts):
    enc, dec = backbone(ids, tok_mask, need_dec=("dec" in parts))
    p = pool_parts(enc, tok_mask, dec)
    return torch.cat([p[k] for k in parts], -1)


class ChunkSeqEncoder(nn.Module):
    """Contextualises the sequence of chunk vectors.
       kind: 'gru' | 'lstm'  -> bidirectional RNN
             'transformer'   -> small BERT-style bidirectional encoder (learned chunk-position embeddings)
             'none'          -> no cross-chunk modelling (ablation baseline)"""

    def __init__(self, d_in, hid=256, kind="gru", layers=1, dropout=0.3, max_chunks=64, heads=4):
        super().__init__()
        self.kind, D = kind, 2 * hid
        self.inp = nn.Sequential(nn.LayerNorm(d_in), nn.Dropout(dropout), nn.Linear(d_in, D), nn.GELU())
        if kind in ("gru", "lstm"):
            self.rnn = _make_rnn(kind, D, hid, layers, dropout)
        elif kind == "transformer":
            self.pos = nn.Embedding(max_chunks, D)
            layer = nn.TransformerEncoderLayer(D, heads, 4 * D, dropout, activation="gelu",
                                               batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        elif kind != "none":
            raise ValueError(kind)
        self.out_dim = D

    def forward(self, x, chunk_mask):                         # [B,N,d_in], [B,N] bool
        z = self.inp(x)
        if self.kind in ("gru", "lstm"):
            z = _pack_run(self.rnn, z, chunk_mask.sum(1))
        elif self.kind == "transformer":
            z = z + self.pos(torch.arange(z.size(1), device=z.device))
            z = self.enc(z, src_key_padding_mask=~chunk_mask)
        return z


class DocHead(nn.Module):
    """Chunk sequence -> one logit per statute.
       'attn'       : one attention over chunks, shared linear classifier (few parameters, safest for rare statutes)
       'label_attn' : every statute has its own attention query and weight vector (self.last_attn [B,L,N] shows
                      which chunk fired for which statute)
       'meanmax'    : masked mean+max pooling"""

    def __init__(self, dim, n_labels, pool="attn", dropout=0.3):
        super().__init__()
        self.pool = pool
        self.drop = nn.Dropout(dropout)
        self.last_attn = None
        if pool == "attn":
            self.att = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))
            self.fc = nn.Linear(dim, n_labels)
        elif pool == "label_attn":
            self.Q = nn.Parameter(torch.empty(n_labels, dim)); nn.init.xavier_uniform_(self.Q)
            self.W = nn.Parameter(torch.empty(n_labels, dim)); nn.init.xavier_uniform_(self.W)
            self.b = nn.Parameter(torch.zeros(n_labels))
        elif pool == "meanmax":
            self.fc = nn.Linear(2 * dim, n_labels)
        else:
            raise ValueError(pool)

    def forward(self, z, mask):                               # [B,N,D], [B,N]
        if self.pool == "attn":
            s = self.att(z).squeeze(-1).masked_fill(~mask, -1e4)
            a = torch.softmax(s, -1)
            self.last_attn = a.detach()
            return self.fc(self.drop((a.unsqueeze(-1) * z).sum(1)))
        if self.pool == "label_attn":
            s = torch.einsum("bnd,ld->bln", z, self.Q).masked_fill(~mask[:, None, :], -1e4)
            a = torch.softmax(s, -1)
            self.last_attn = a.detach()
            ctx = torch.einsum("bln,bnd->bld", a, z)
            return (self.drop(ctx) * self.W).sum(-1) + self.b
        m = mask.unsqueeze(-1)
        mean = (z * m).sum(1) / m.sum(1)
        mx = z.masked_fill(~m, -1e4).max(1).values
        return self.fc(self.drop(torch.cat([mean, mx], -1)))


class EDTokenClassifier(nn.Module):
    """MODE 'token': the backbone runs on the fly (frozen, adapted by cfg.peft, or with the top-k encoder layers / decoder
       fine-tuned), in micro-batches of chunks, then chunk-level encoder -> statute head."""

    def __init__(self, ed, n_labels, cfg):
        super().__init__()
        self.parts = parse_parts(cfg.parts)
        self.backbone = EDBackbone(ed, cfg.unfreeze_top_k, cfg.unfreeze_decoder, cfg.use_layer)
        # PEFT / fine-tuning method for this experiment (None = frozen backbone). See cell 'PEFT for the encoder-decoder'.
        self.peft = apply_peft(self.backbone, cfg.peft) if getattr(cfg, "peft", None) else None
        d_in = len(self.parts) * self.backbone.d_model
        self.chunk = ChunkSeqEncoder(d_in, cfg.chunk_hid, cfg.chunk_rnn, cfg.chunk_layers, cfg.dropout, cfg.max_chunks)
        self.head = DocHead(self.chunk.out_dim, n_labels, cfg.pool, cfg.dropout)
        self.chunk_bs = cfg.ed_chunk_bs

    def forward(self, ids, tok_mask, chunk_mask):             # [B,N,T], [B,N,T], [B,N]
        B, N, T = ids.shape
        flat_ids, flat_tm = ids[chunk_mask], tok_mask[chunk_mask]        # only real chunks: [M,T]
        vecs = []
        for i in range(0, flat_ids.size(0), self.chunk_bs):
            tm = flat_tm[i:i + self.chunk_bs]
            t = int(tm.sum(1).max())                                      # trim padding to this micro-batch's longest chunk
            vecs.append(chunk_vectors(self.backbone, flat_ids[i:i + self.chunk_bs, :t], tm[:, :t], self.parts))
        vecs = torch.cat(vecs)
        x = vecs.new_zeros(B, N, vecs.size(-1))
        x[chunk_mask] = vecs
        return self.head(self.chunk(x, chunk_mask), chunk_mask)


class CachedFeatureClassifier(nn.Module):
    """MODE 'cached': backbone frozen and run ONCE; every chunk is stored as [mean | max | eos | dec] (fp16).
       Only the chunk-level encoder + head are trained -> seconds per epoch, ideal for ablations."""

    def __init__(self, in_dim, n_labels, cfg):
        super().__init__()
        self.chunk = ChunkSeqEncoder(in_dim, cfg.chunk_hid, cfg.chunk_rnn, cfg.chunk_layers, cfg.dropout, cfg.max_chunks)
        self.head = DocHead(self.chunk.out_dim, n_labels, cfg.pool, cfg.dropout)

    def forward(self, feats, chunk_mask):
        return self.head(self.chunk(feats, chunk_mask), chunk_mask)

In [ ]:
# ============================================================
# 4. DATASETS, COLLATE, FAST FEATURE EXTRACTION
# ============================================================
import shutil


def pick_amp_dtype():
    if device.type != "cuda":
        return None
    # bf16 only on Ampere+ (a T4 would emulate it slowly) -> fp16 on T4/V100
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


AMP_DTYPE = pick_amp_dtype()
print("mixed-precision dtype:", AMP_DTYPE)


class ChunkedDocs(Dataset):
    """Each document -> list of <=511-token chunk VIEWS (after the long-document strategy). The <eos> is appended
    when a batch is built, so no per-chunk copies are stored."""

    def __init__(self, enc_docs, y, cfg):
        body = cfg.chunk_len - 1
        self.items = [select_chunks(chunk_ids(d, body, cfg.chunk_stride), cfg.max_chunks, cfg.chunk_strategy)
                      for d in enc_docs]
        self.y = y

    def __len__(self): return len(self.items)

    def __getitem__(self, i): return i, self.items[i], (None if self.y is None else self.y[i])


class FeatDocs(Dataset):
    def __init__(self, feats, y):
        self.feats, self.y = feats, y

    def __len__(self): return len(self.feats)

    def __getitem__(self, i): return i, self.feats[i], (None if self.y is None else self.y[i])


def _stack_y(batch):
    return None if batch[0][2] is None else torch.from_numpy(np.stack([b[2] for b in batch]))


def collate_tokens(batch):
    B = len(batch)
    N = max(len(c) for _, c, _ in batch)
    T = max(len(x) for _, c, _ in batch for x in c) + 1                   # +1 for <eos>
    ids = torch.full((B, N, T), PAD_ID, dtype=torch.long)
    tm = torch.zeros(B, N, T, dtype=torch.bool)
    cm = torch.zeros(B, N, dtype=torch.bool)
    for b, (_, chunks, _) in enumerate(batch):
        for n, c in enumerate(chunks):
            k = len(c)
            ids[b, n, :k] = torch.from_numpy(np.asarray(c, dtype=np.int64))
            ids[b, n, k] = EOS_ID
            tm[b, n, :k + 1] = True
            cm[b, n] = True
    return dict(ids=ids, tok_mask=tm, chunk_mask=cm, y=_stack_y(batch), idx=torch.tensor([b[0] for b in batch]))   # idx: doc index (KD teacher logits)


def feat_columns(parts, d):
    """column indices of the chosen parts inside the cached [mean|max|eos|dec] vector"""
    return [i * d + j for i, name in enumerate(PART_NAMES) if name in parts for j in range(d)]


def make_collate_feats(cols):
    cols = torch.as_tensor(cols, dtype=torch.long)

    def collate_feats(batch):
        B, N = len(batch), max(f.size(0) for _, f, _ in batch)
        x = torch.zeros(B, N, len(cols))
        cm = torch.zeros(B, N, dtype=torch.bool)
        for b, (_, f, _) in enumerate(batch):
            x[b, :f.size(0)] = f.index_select(1, cols).float()
            cm[b, :f.size(0)] = True
        return dict(feats=x, chunk_mask=cm, y=_stack_y(batch), idx=torch.tensor([b[0] for b in batch]))
    return collate_feats


@torch.no_grad()
def extract_chunk_features(ed, ds, tok_budget=32768, max_bs=256, use_layer=-1, amp=True, log_every=25):
    """Run the frozen encoder-decoder ONCE over every chunk -> fp16 [N_i, 4*d] per document: [mean|max|eos|dec].
    Batches are built from a TOKEN budget over length-sorted chunks (short chunks -> bigger batches), under fp16/bf16
    autocast with fused attention. Right-padding is safe: padded keys are masked out of the encoder's attention.
    use_layer=k>=0 -> truncated encoder (k+1 layers, the distillation student); its 'dec' columns are left at zero (the
    decoder was trained on the output of ALL encoder layers, so it is not run on shallower states)."""
    bb = EDBackbone(ed, 0, False, use_layer).to(device).eval()
    full = use_layer == -1
    flat = [c for chunks in ds.items for c in chunks]                     # doc-major order
    doc_len = [len(chunks) for chunks in ds.items]
    n, d = len(flat), bb.d_model
    lens = np.fromiter((len(c) + 1 for c in flat), dtype=np.int64, count=n)   # +1 for <eos>
    order = np.argsort(lens, kind="stable")                               # similar lengths -> little padding
    out = torch.empty(n, len(PART_NAMES) * d, dtype=torch.float16)        # one preallocated buffer
    use_amp = amp and AMP_DTYPE is not None
    print(f"  {n:,} chunks | {int(lens.sum()):,} tokens | amp={AMP_DTYPE if use_amp else 'off'} | budget {tok_budget} tok/batch")

    def run(sel):
        T = int(lens[sel].max())
        arr = np.full((len(sel), T), PAD_ID, dtype=np.int64)
        for j, i in enumerate(sel):
            k = lens[i] - 1
            arr[j, :k] = flat[i]
            arr[j, k] = EOS_ID
        ids = torch.from_numpy(arr).to(device, non_blocking=True)
        ln = torch.from_numpy(lens[sel]).to(device)
        tm = torch.arange(T, device=device)[None, :] < ln[:, None]
        with torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=use_amp):
            enc, dec = bb(ids, tm, need_dec=full)
        p = pool_parts(enc, tm, dec)
        if not full:
            p["dec"] = torch.zeros_like(p["mean"])
        out[torch.from_numpy(sel)] = torch.cat([p[k] for k in PART_NAMES], -1).half().cpu()

    def safe_run(sel):                                                    # halve the batch on CUDA OOM
        try:
            run(sel)
        except torch.cuda.OutOfMemoryError:
            if len(sel) == 1:
                raise
            torch.cuda.empty_cache()
            mid = len(sel) // 2
            safe_run(sel[:mid]); safe_run(sel[mid:])

    t0, done_tok, nb, s = time.time(), 0, 0, 0
    while s < n:
        e = s + 1
        while e < n and (e - s) < max_bs and (e + 1 - s) * lens[order[e]] <= tok_budget:
            e += 1
        sel = order[s:e]
        safe_run(sel)
        done_tok += int(lens[sel].sum()); s = e; nb += 1
        if nb % log_every == 0 or s >= n:
            dt = time.time() - t0
            eta = dt * (int(lens.sum()) - done_tok) / max(done_tok, 1)
            print(f"  features {s:,}/{n:,} chunks | {done_tok/dt:,.0f} tok/s | elapsed {dt/60:.1f} min | ETA {eta/60:.1f} min")

    feats, p = [], 0
    for k in doc_len:                                                     # per-doc views of the one buffer
        feats.append(out[p:p + k]); p += k
    return feats


def get_features(split, enc_docs, cfg):
    """Cached per (split, chunking). One cache serves every `parts` ablation (all four parts are stored)."""
    # checkpoint signature + encoder depth in the name: can never be confused with features of another checkpoint / student depth
    sig = f"ed{CKPT_SIG}_{split}_L{cfg.chunk_len}_S{cfg.chunk_stride}_K{cfg.max_chunks}_{cfg.chunk_strategy}_layer{cfg.use_layer}"
    name = f"edfeat_{sig}.pt"
    drive_path = os.path.join(p5_dir, name)
    local_dir = "/content/feat_cache"; os.makedirs(local_dir, exist_ok=True)     # local disk is much faster than Drive
    local_path = os.path.join(local_dir, name)
    if not os.path.exists(local_path) and os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
    if os.path.exists(local_path):
        return torch.load(local_path, weights_only=False)
    print(f"extracting encoder-decoder features for '{split}' ...")
    ds = ChunkedDocs(enc_docs, None, cfg)
    feats = extract_chunk_features(ED.to(device), ds, use_layer=cfg.use_layer)
    torch.save(feats, local_path)
    shutil.copy(local_path, drive_path)                                          # persist to Drive once
    return feats

mixed-precision dtype: torch.float16


In [ ]:
# ============================================================
# 4b. SELF-TEST of the fast path -- run once, takes seconds. Proves that fused attention + mixed precision give the
#     same features as the original manual attention, on REAL chunks of different lengths (so padding is exercised).
# ============================================================
@torch.no_grad()
def check_fast_path():
    cuts = [511, 400, 300, 200, 128, 64, 511, 350]
    docs = [d for d in enc_train if len(d) >= 511][:len(cuts)]
    assert docs, "no training document has >= 511 tokens"
    chunks = [d[:c] for d, c in zip(docs, cuts)]
    B, T = len(chunks), max(len(c) for c in chunks) + 1
    arr = np.full((B, T), PAD_ID, dtype=np.int64)
    ln = torch.tensor([len(c) + 1 for c in chunks], device=device)
    for j, c in enumerate(chunks):
        arr[j, :len(c)] = c; arr[j, len(c)] = EOS_ID
    ids = torch.from_numpy(arr).to(device)
    tm = torch.arange(T, device=device)[None, :] < ln[:, None]
    bb = EDBackbone(ED.to(device), 0, False).to(device).eval()

    def timed(fast, amp):
        FAST_ATTN["on"] = fast
        if device.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        with torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=amp and AMP_DTYPE is not None):
            enc, dec = bb(ids, tm, need_dec=True)
        if device.type == "cuda": torch.cuda.synchronize()
        return enc.float(), dec.float(), time.time() - t

    timed(True, False)                                            # warm-up (kernel selection / allocator)
    e0, d0, t0 = timed(False, False)                              # original attention, fp32
    e1, d1, t1 = timed(True, False)                               # fused attention, fp32
    e2, d2, t2 = timed(True, True)                                # fused attention + autocast
    FAST_ATTN["on"] = True

    m = tm.unsqueeze(-1)
    err = lambda a, b: float(((a - b).abs() * m).max())
    cos = lambda a, b: float(F.cosine_similarity(a[tm], b[tm], dim=-1).min())
    print(f"original fp32   : {t0*1000:7.0f} ms")
    print(f"fused fp32      : {t1*1000:7.0f} ms | enc max|diff| {err(e0, e1):.2e} | dec max|diff| {(d0-d1).abs().max():.2e}")
    print(f"fused + autocast: {t2*1000:7.0f} ms | enc min cos {cos(e0, e2):.5f} | dec min cos "
          f"{F.cosine_similarity(d0, d2, dim=-1).min():.5f}")
    print("NaN/Inf in autocast output:", bool((~torch.isfinite(e2)).any() or (~torch.isfinite(d2)).any()))
    scale = max(1.0, float((e0.abs() * m).max()), float(d0.abs().max()))      # outputs are RMS-normed * learned weight
    assert err(e0, e1) < 1e-3 * scale and (d0 - d1).abs().max() < 1e-3 * scale, \
        "fused attention does not match the original attention!"
    if cos(e0, e2) < 0.99 or not torch.isfinite(e2).all():
        print("WARNING: mixed precision hurts the features -> pass amp=False to extract_chunk_features")


check_fast_path()

original fp32   :     266 ms
fused fp32      :     131 ms | enc max|diff| 2.26e-06 | dec max|diff| 4.77e-07
fused + autocast:     263 ms | enc min cos 1.00000 | dec min cos 1.00000
NaN/Inf in autocast output: False


In [ ]:
# ============================================================
# 4c. SELF-TEST of the PEFT modules -- run once, takes seconds. Every method must start as the IDENTITY on the pre-trained
#     encoder (so training begins from the Phase-1 model): features with the method injected == features without it.
#     Exceptions by design: QLoRA (4-bit rounding changes the weights slightly), prompt tuning (adds tokens) and prefix tuning
#     (the extra keys take a small share of every softmax) -- they are printed, not asserted.
# ============================================================
@torch.no_grad()
def check_peft_identity():
    docs = [d for d in enc_train if len(d) >= 200][:4]
    chunks = [d[:200] for d in docs]
    ids = torch.tensor([c + [EOS_ID] for c in chunks], device=device)
    tm = torch.ones_like(ids, dtype=torch.bool)

    def feats(bb):
        bb.eval()
        enc, _ = bb(ids, tm)
        return pool_parts(enc, tm)["mean"]

    ref = feats(EDBackbone(ED.to(device), 0, False))
    print(f"{'method':12s} {'new params':>11s} {'max|diff| vs pre-trained':>26s}")
    for spec in (dict(method="lorma_plus"), dict(method="lorma_pi"), dict(method="adalora"), dict(method="adapter"),
                 dict(method="prefix"), dict(method="qlora"), dict(method="prompt")):
        bb = EDBackbone(copy.deepcopy(ED), 0, False).to(device)
        h = apply_peft(bb, spec)
        d = float((feats(bb) - ref).abs().max())
        n = sum(p.numel() for p in h.params)
        flag = "" if spec["method"] in ("qlora", "prompt", "prefix") else ("   OK" if d < 1e-4 else "   <-- NOT identity!")
        print(f"{spec['method']:12s} {n:11,d} {d:26.2e}{flag}")
        if spec["method"] not in ("qlora", "prompt", "prefix"):
            assert d < 1e-4, f"{spec['method']} is not the identity at initialisation"
        del bb
    if device.type == "cuda":
        torch.cuda.empty_cache()


check_peft_identity()


method        new params   max|diff| vs pre-trained
lorma_plus        65,536                   0.00e+00   OK


/tmp/ipykernel_844/121545631.py:11: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  ids = torch.tensor([c + [EOS_ID] for c in chunks], device=device)


lorma_pi          65,536                   0.00e+00   OK
adalora          565,584                   0.00e+00   OK
adapter          528,896                   0.00e+00   OK
prefix            65,536                   6.82e-02
qlora            753,664                   1.37e-01
prompt            10,240                   1.35e-01


In [ ]:
# ============================================================
# 5. LOSSES, TRAINING, EVALUATION   (as in the GQLA notebook; adds PEFT optimiser groups, knowledge distillation and cost tracking)
# ============================================================
class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021): down-weights the many easy negatives of a multi-label problem."""

    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits)
        xs_neg = (1 - p + self.clip).clamp(max=1) if self.clip > 0 else 1 - p
        los = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        pt = p * y + xs_neg * (1 - y)
        gamma = self.gp * y + self.gn * (1 - y)
        return -(los * (1 - pt) ** gamma).sum() / logits.size(0)


def build_loss(cfg, y_tr):
    if cfg.loss == "bce":
        return nn.BCEWithLogitsLoss()
    if cfg.loss == "bce_pw":      # rare statutes get a larger positive weight: ((N-n_c)/n_c)^power, clipped
        pos = y_tr.sum(0)
        pw = np.clip(((len(y_tr) - pos) / np.maximum(pos, 1)) ** cfg.pw_power, 1.0, cfg.pw_max)
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, dtype=torch.float32))
    if cfg.loss == "asl":
        return AsymmetricLoss()
    raise ValueError(cfg.loss)


def forward_batch(model, batch, mode):
    cm = batch["chunk_mask"].to(device)
    if mode == "token":
        return model(batch["ids"].to(device), batch["tok_mask"].to(device), cm)
    return model(batch["feats"].to(device), cm)


def resolve_amp(cfg, mode):
    """cfg.amp: None -> on for 'token' mode (the encoder forward runs every step), off for 'cached' (tiny head);
    True/False forces it. Always off when the self-test disabled half precision or there is no GPU."""
    want = (mode == "token") if getattr(cfg, "amp", None) is None else bool(cfg.amp)
    return want and AMP_DTYPE is not None and device.type == "cuda"


def amp_context(enabled):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=enabled)


@torch.no_grad()
def predict_logits(model, loader, mode, amp=False):
    model.eval()
    out = []
    for b in loader:
        with amp_context(amp):
            logits = forward_batch(model, b, mode)
        out.append(logits.float().cpu().numpy())
    return np.concatenate(out)


def predict_probs(model, loader, mode, amp=False):
    return 1.0 / (1.0 + np.exp(-np.clip(predict_logits(model, loader, mode, amp), -30, 30)))


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def train_model(model, mode, train_loader, dev_loader, y_dev, cfg, y_tr, tag):
    loss_fn = build_loss(cfg, y_tr).to(device)
    peft = getattr(model, "peft", None)
    peft_ids = peft.param_ids() if peft is not None else set()
    new_p = [p for n, p in model.named_parameters() if p.requires_grad and id(p) in peft_ids]       # new PEFT params
    bb = [p for n, p in model.named_parameters()
          if p.requires_grad and n.startswith("backbone.") and id(p) not in peft_ids]               # pre-trained weights being tuned
    rest = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]  # chunk-level head
    groups = ([dict(params=rest, lr=cfg.lr)] + ([dict(params=new_p, lr=cfg.lr_peft)] if new_p else [])
              + ([dict(params=bb, lr=cfg.lr_backbone)] if bb else []))
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd)
    total = cfg.epochs * math.ceil(len(train_loader) / cfg.accum)
    if peft is not None:
        peft.setup(total)
        print(f"[{tag}] {peft.describe()}")
    kd = getattr(cfg, "kd", None)                       # knowledge distillation: dict(teachers=[names], T=2.0, lam=0.5)
    if kd:
        t_logits = torch.from_numpy(cfg.kd_teacher_logits)          # [N_train, L], same order as the training docs
        print(f"[{tag}] distilling from {kd['teachers']} | T={kd['T']} lambda={kd['lam']}")
    gstep = 0
    warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))

    use_amp = resolve_amp(cfg, mode)
    # fp16 needs loss scaling; bf16 (Ampere+) does not. Disabled scaler = plain backward/step.
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and AMP_DTYPE == torch.float16)
    print(f"[{tag}] autocast: {AMP_DTYPE if use_amp else 'off'} | grad scaler: {scaler.is_enabled()}")

    best, best_state, bad = -1.0, None, 0
    for ep in range(cfg.epochs):
        model.train()
        t0, run, step = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, b in enumerate(train_loader):
            with amp_context(use_amp):
                logits = forward_batch(model, b, mode)
            logits, y = logits.float(), b["y"].to(device)               # losses always in fp32
            loss = loss_fn(logits, y)
            if kd:                                                      # (1-lam) * hard + lam * T^2 * soft (sigmoid targets)
                T = kd["T"]
                tl = t_logits[b["idx"]].to(device)
                soft = F.binary_cross_entropy_with_logits(logits / T, torch.sigmoid(tl / T)) * (T * T)
                loss = (1 - kd["lam"]) * loss + kd["lam"] * soft
            if peft is not None:
                loss = loss + peft.extra_loss()                         # AdaLoRA orthogonality regulariser
            scaler.scale(loss / cfg.accum).backward()
            run += loss.item()
            if (i + 1) % cfg.accum == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(opt)
                if peft is not None:
                    peft.after_backward(gstep)                          # AdaLoRA: update importance scores
                nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
                gstep += 1
                if peft is not None:
                    peft.after_step(gstep)                              # AdaLoRA: prune ranks to the budget schedule
            if ep == 0 and i + 1 == 20:                                 # cost estimate after 20 mini-batches
                sp = (time.time() - t0) / 20
                print(f"[{tag}] {sp:.2f} s/batch -> ~{sp*len(train_loader)/60:.1f} min per epoch")
        probs = predict_probs(model, dev_loader, mode, use_amp)
        thr, macro = tune_global_threshold(probs, y_dev)
        m = compute_metrics(y_dev, predict_labels(probs, thr))
        print(f"[{tag}] ep {ep+1:02d}  loss {run/len(train_loader):.4f}  dev macro-F1 {macro:.4f}  "
              f"micro-F1 {m['micro_f1']:.4f}  subset {m['subset_acc']:.4f}  thr {thr:.3f}  ({time.time()-t0:.0f}s)")
        if macro > best:
            best, bad = macro, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
    model.load_state_dict(best_state, strict=False)           # frozen encoder-decoder weights are untouched
    return best_state


BASE = dict(
    # long-document representation: chunk = 511 tokens + <eos> = 512 (the Phase-1 encoder window)
    chunk_len=512, chunk_stride=511, max_chunks=16, chunk_strategy="head_tail",
    # chunk vector = concatenation of these parts of the backbone: mean | max | eos | dec  (joined by '+')
    parts="mean+max+eos",
    # backbone: use_layer=-1 -> all 9 encoder layers; k>=0 -> truncated encoder of k+1 layers (distillation student)
    use_layer=-1,
    # token mode only: legacy partial fine-tuning switches (peft=dict(method="partial", k=..) does the same), micro-batch of chunks
    unfreeze_top_k=0, unfreeze_decoder=False, ed_chunk_bs=8,
    # chunk-level bidirectional head
    chunk_rnn="gru", chunk_hid=256, chunk_layers=1, pool="attn", dropout=0.3,
    # loss
    loss="bce_pw", pw_power=0.5, pw_max=30.0,
    # optimisation
    lr=1e-3, lr_backbone=2e-5, lr_peft=5e-4, wd=0.01, epochs=20, patience=5, batch_size=32, accum=1, warmup=0.06, clip=1.0,
    seed=42, min_support=3,
    # mixed precision: None = auto (on for mode='token', off for 'cached'); True/False forces it
    amp=None,
    # adaptation method for mode='token' (see 'PEFT for the encoder-decoder'): None | dict(method=..., **overrides);
    # distillation: dict(teachers=[...], T=2.0, lam=0.5)
    peft=None, kd=None,
)

RESULTS = {}


def run_experiment(name, mode="cached", keep_train_logits=False, **over):
    """mode='cached' : encoder-decoder run once -> train the chunk-level BiRNN over cached chunk vectors (fast; no adaptation).
       mode='token'  : backbone run on the fly -> chunk-level BiRNN; cfg.peft adapts the encoder itself.
       keep_train_logits=True stores the model's logits on the TRAIN documents (needed if it will be a KD teacher)."""
    cfg = SimpleNamespace(**{**BASE, **over})
    parts = parse_parts(cfg.parts)
    set_seed(cfg.seed)
    if cfg.peft and mode != "token":
        raise ValueError("peft adapts the encoder itself -> use mode='token'")
    if mode == "cached":
        ftr, fdv, fte = (get_features(s, e, cfg) for s, e in
                         (("train", enc_train), ("dev", enc_dev), ("test", enc_test)))
        cols = feat_columns(parts, D_ED)
        d_tr, d_dv, d_te = FeatDocs(ftr, y_train), FeatDocs(fdv, y_dev), FeatDocs(fte, None)
        coll, model = make_collate_feats(cols), CachedFeatureClassifier(len(cols), L, cfg)
        g = None
    else:
        d_tr, d_dv, d_te = (ChunkedDocs(enc_train, y_train, cfg), ChunkedDocs(enc_dev, y_dev, cfg),
                            ChunkedDocs(enc_test, None, cfg))
        finetune = bool(cfg.peft) or cfg.unfreeze_top_k > 0 or cfg.unfreeze_decoder
        g = copy.deepcopy(ED) if finetune else ED             # never modify the shared Phase-1 model
        coll, model = collate_tokens, EDTokenClassifier(g, L, cfg)
    if cfg.kd:                                                       # teacher logits on the training docs (averaged)
        miss = [t for t in cfg.kd["teachers"] if "train_logits" not in RESULTS.get(t, {})]
        assert not miss, f"teacher(s) {miss}: run them first with keep_train_logits=True"
        cfg.kd_teacher_logits = np.mean([RESULTS[t]["train_logits"] for t in cfg.kd["teachers"]], axis=0).astype(np.float32)
    model.to(device)
    mk = lambda d, sh: DataLoader(d, batch_size=cfg.batch_size if sh else max(cfg.batch_size, 8), shuffle=sh,
                                  collate_fn=coll, num_workers=0, drop_last=False)
    tr, dv, te, tr_eval = mk(d_tr, True), mk(d_dv, False), mk(d_te, False), mk(d_tr, False)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_used = model.backbone.n_used if mode == "token" else (len(ED.encoder.layers) if cfg.use_layer == -1 else cfg.use_layer + 1)
    params_M, size_MB = model_footprint(model, mode, n_used, parts)
    print(f"\n=== {name} | mode={mode} | parts={'+'.join(parts)} | trainable params {n_train/1e6:.3f}M | deployed pipeline "
          f"{params_M:.1f}M params / {size_MB:.0f} MB ({n_used} encoder layers) ===")

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_model(model, mode, tr, dv, y_dev, cfg, y_train, name)
    train_s = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0

    amp_on = resolve_amp(cfg, mode)
    t0 = time.time()
    lg_dev = predict_logits(model, dv, mode, amp_on)
    infer_s = time.time() - t0                                        # dev-set inference wall-clock (incl. the encoder forward in token mode)
    p_dev = 1.0 / (1.0 + np.exp(-np.clip(lg_dev, -30, 30)))
    p_test = predict_probs(model, te, mode, amp_on)
    g_thr, _ = tune_global_threshold(p_dev, y_dev)
    pc_thr = tune_per_class_threshold(p_dev, y_dev, g_thr, min_support=cfg.min_support)
    res = dict(cfg={k: v for k, v in vars(cfg).items() if k != "kd_teacher_logits"}, mode=mode, p_dev=p_dev, p_test=p_test,
               g_thr=g_thr, pc_thr=pc_thr, trainable_params=n_train, deployed_params_M=params_M, deployed_size_MB=size_MB,
               train_seconds=train_s, dev_infer_seconds=infer_s, peak_gpu_MB=peak_mb,
               m_fixed=compute_metrics(y_dev, predict_labels(p_dev, 0.5)),
               m_global=compute_metrics(y_dev, predict_labels(p_dev, g_thr)),
               m_perclass=compute_metrics(y_dev, predict_labels(p_dev, pc_thr)),
               honest=honest_threshold_eval(p_dev, y_dev))
    if getattr(model, "peft", None) is not None and model.peft.allocator is not None:
        res["adalora_ranks"] = model.peft.allocator.effective_ranks()
    if keep_train_logits:
        res["train_logits"] = predict_logits(model, tr_eval, mode, amp_on)
    RESULTS[name] = res
    pickle.dump({k: v for k, v in res.items()}, open(os.path.join(p5_dir, f"result_{name}.pkl"), "wb"))
    print(f"[{name}] dev macro-F1  thr=0.5: {res['m_fixed']['macro_f1']:.4f} | global thr {g_thr:.3f}: "
          f"{res['m_global']['macro_f1']:.4f} | per-class: {res['m_perclass']['macro_f1']:.4f} (optimistic)")
    print(f"[{name}] honest split-half macro-F1 (tune on half, score on other half): {res['honest']}")
    print(f"[{name}] cost: train {train_s/60:.1f} min | dev inference {infer_s:.1f}s | peak GPU {peak_mb:.0f} MB")
    del model, g
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return res


def write_submission(name, path, per_class=False):
    r = RESULTS[name]
    thr = r["pc_thr"] if per_class else r["g_thr"]
    pred = predict_labels(r["p_test"], thr)
    out = [dict(id=rec["id"], pred=sorted(int(idx2lab[j]) for j in np.flatnonzero(p)))
           for rec, p in zip(test_recs, pred)]
    assert len(out) == len(test_recs) == len({o["id"] for o in out}), "need exactly one prediction per test id"
    assert all(len(o["pred"]) >= 1 and len(set(o["pred"])) == len(o["pred"]) for o in out)
    json.dump(out, open(path, "w"), ensure_ascii=False)
    print(f"wrote {len(out)} predictions -> {path}  (avg {np.mean([len(o['pred']) for o in out]):.2f} statutes/doc)")

In [ ]:
# ============================================================
# 6. RUN  -  the six Phase-5 experiments for the Encoder-Decoder architecture
#
#   baseline  frozen encoder-decoder + BiGRU on cached features     (zero adaptation of the backbone; NOT one of the six)
#   E1  LoRMA+            multiplicative low-rank adaptation of the q and v matrices        [compulsory technique]
#   E2  AdaLoRA           SVD-style low-rank update with importance-based rank pruning
#   E3  QLoRA             frozen 4-bit NF4 encoder + LoRA
#   E4  Adapters          bottleneck adapters after attention and after the FFN (Houlsby)
#   E5  Partial FT        fine-tune the top-3 encoder layers (layer-wise fine-tuning)
#   E6  Knowledge dist.   teacher = ensemble of E1-E5 (9-layer encoder + adaptation), student = 4-layer encoder + small BiGRU
#
# That is 5 different fine-tuning / PEFT techniques + 1 distillation experiment (brief, sec. 11.2).
# Optional extras below: LoRMA-pi, prefix tuning, prompt tuning, full fine-tuning, student without distillation.
#
# !! Every number below (epochs, ranks, learning rates, chunks, temperature, ...) is only a STARTING POINT so that the
# !! notebook runs end-to-end. Tuning them and interpreting the results is your job (and the course rules say so).
# !! Run ONE experiment per cell first and read the "s/batch -> ~N min per epoch" line before queueing the rest.
# ============================================================
TOK = dict(mode="token", batch_size=4, accum=4, max_chunks=8, epochs=4, patience=2,
           lr=1e-3, lr_peft=5e-4, lr_backbone=2e-5, ed_chunk_bs=8)

# ---- baseline: frozen encoder-decoder, cached features (fast; shows what the PEFT methods have to beat) -----------------
r_base = run_experiment("cached_bigru", mode="cached")

# ---- E1  LoRMA+  (compulsory) -------------------------------------------------------------------------------------
r1 = run_experiment("e1_lorma_plus", keep_train_logits=True,
                    peft=dict(method="lorma_plus", r=8, alpha=16, top_layers=4, targets=("q", "v")), **TOK)

# ---- E2  AdaLoRA --------------------------------------------------------------------------------------------------
r2 = run_experiment("e2_adalora", keep_train_logits=True,
                    peft=dict(method="adalora", init_r=12, target_r=6, top_layers=4), **TOK)

# ---- E3  QLoRA  (4-bit NF4 base for all 9 encoder layers, LoRA in the top 4 layers) ----------------------------------------
r3 = run_experiment("e3_qlora", keep_train_logits=True,
                    peft=dict(method="qlora", r=16, alpha=32, top_layers=4), **TOK)

# ---- E4  Bottleneck adapters ----------------------------------------------------------------------------------------
r4 = run_experiment("e4_adapter", keep_train_logits=True,
                    peft=dict(method="adapter", bottleneck=64, top_layers=4), **TOK)

# ---- E5  Partial / layer-wise fine-tuning (pre-trained weights of the top 3 encoder layers + final norm are updated) -------------------------
r5 = run_experiment("e5_partial_ft", keep_train_logits=True,
                    peft=dict(method="partial", k=3), **TOK)

# ---- E6  Knowledge distillation -------------------------------------------------------------------------------------
# Teacher : ensemble (mean logit) of E1-E5 = 9 encoder layers + adaptation + hierarchical BiGRU head.
# Student : the FIRST 4 encoder layers (use_layer=3, parameter-free RMS norm on top) frozen + a smaller BiGRU head (chunk_hid=128);
#           features cached; the decoder is not used (parts = mean+max+eos). -> 4/9 of the encoder compute and size at inference.
#           Loss = (1-lam)*weighted-BCE + lam*T^2*BCE(student/T, sigmoid(teacher/T)).
r6 = run_experiment("e6_kd_student", mode="cached", use_layer=3, chunk_hid=128,
                    kd=dict(teachers=["e1_lorma_plus", "e2_adalora", "e3_qlora", "e4_adapter", "e5_partial_ft"], T=2.0, lam=0.5))

# ---- optional extras --------------------------------------------------------------------------------------------------
# r6b = run_experiment("e6b_student_no_kd", mode="cached", use_layer=3, chunk_hid=128)        # same student, no teacher (isolates the KD effect)
# r7  = run_experiment("x_lorma_pi", keep_train_logits=True, peft=dict(method="lorma_pi", r=8, top_layers=4), **TOK)
# r7b = run_experiment("x_prefix", peft=dict(method="prefix", n_tokens=16, top_layers=4), **{**TOK, "lr_peft": 5e-3})
# r8  = run_experiment("x_prompt", peft=dict(method="prompt", n_tokens=20), **{**TOK, "lr_peft": 5e-2})     # backward through ALL 9 layers: slow
# r9  = run_experiment("x_full_ft", peft=dict(method="full"), **{**TOK, "batch_size": 2, "accum": 8})        # whole encoder + embeddings: slow, needs memory
# Other levers from the brief on the same code: loss=("bce"|"bce_pw"|"asl"), chunk_strategy, max_chunks, pool="label_attn",
# chunk_rnn=("gru"|"lstm"|"transformer"), parts (mean|max|eos|dec), use_layer, and the thresholds (global / per class) in the summary below.

# ---- summary table (accuracy + cost) ------------------------------------------------------------------------------------------
print(f"\n{'experiment':20s} {'macro-F1':>8s} {'micro-F1':>8s} {'subset':>7s} {'honest(g)':>9s} {'train M':>8s} "
      f"{'deploy M':>8s} {'MB':>6s} {'train min':>9s} {'dev inf s':>9s} {'peak MB':>8s}")
for n, r in RESULTS.items():
    m = r["m_global"]
    print(f"{n:20s} {m['macro_f1']:8.4f} {m['micro_f1']:8.4f} {m['subset_acc']:7.4f} {r['honest']['global']:9.4f} "
          f"{r['trainable_params']/1e6:8.3f} {r['deployed_params_M']:8.1f} {r['deployed_size_MB']:6.0f} "
          f"{r['train_seconds']/60:9.1f} {r['dev_infer_seconds']:9.1f} {r['peak_gpu_MB']:8.0f}")
if "e2_adalora" in RESULTS:
    print("AdaLoRA final ranks per adapted matrix:", RESULTS["e2_adalora"]["adalora_ranks"])

# ---- rare-statute analysis for one run ----------------------------------------------------------------------------------
best_name = max(RESULTS, key=lambda n: RESULTS[n]["honest"]["global"])
r = RESULTS[best_name]
print(f"\nF1 by number of TRAIN docs per statute ({best_name}):")
for bucket, n_cls, f1 in rare_report(y_dev, predict_labels(r["p_dev"], r["g_thr"]), y_train):
    print(f"  train-freq {bucket:>8s}: {n_cls:4d} statutes  mean F1 {f1:.3f}")


# ---- Codabench packages: prediction.json at the ROOT of submission_<name>.zip -------------------------------------------------
import zipfile


def make_submission_zip(name, per_class=None):
    """Writes prediction.json for experiment `name` and zips it (flat, as Codabench expects).
       per_class=None -> use per-class thresholds only if their honest split-half score beats the global threshold."""
    r = RESULTS[name]
    use_pc = (r["honest"]["per_class"] > r["honest"]["global"]) if per_class is None else per_class
    d = os.path.join(p5_dir, f"sub_{name}"); os.makedirs(d, exist_ok=True)
    write_submission(name, os.path.join(d, "prediction.json"), per_class=use_pc)
    zpath = os.path.join(p5_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(d, "prediction.json"), arcname="prediction.json")
    print(f"  -> {zpath}")
    return zpath


# One zip per experiment (the brief needs >= 6 distinct, non-identical submissions):
for name in RESULTS:
    make_submission_zip(name)

extracting encoder-decoder features for 'train' ...
  147,302 chunks | 64,501,705 tokens | amp=torch.float16 | budget 32768 tok/batch
  features 6,400/147,302 chunks | 117,705 tok/s | elapsed 0.0 min | ETA 9.1 min
  features 12,503/147,302 chunks | 118,391 tok/s | elapsed 0.2 min | ETA 8.9 min
  features 17,055/147,302 chunks | 115,857 tok/s | elapsed 0.3 min | ETA 9.0 min
  features 20,709/147,302 chunks | 112,528 tok/s | elapsed 0.4 min | ETA 9.2 min
  features 23,842/147,302 chunks | 110,120 tok/s | elapsed 0.5 min | ETA 9.2 min
  features 26,615/147,302 chunks | 107,711 tok/s | elapsed 0.7 min | ETA 9.3 min
  features 29,122/147,302 chunks | 105,210 tok/s | elapsed 0.8 min | ETA 9.4 min
  features 31,423/147,302 chunks | 102,698 tok/s | elapsed 1.0 min | ETA 9.5 min
  features 33,558/147,302 chunks | 100,237 tok/s | elapsed 1.1 min | ETA 9.6 min
  features 35,553/147,302 chunks | 97,642 tok/s | elapsed 1.3 min | ETA 9.7 min
  features 37,424/147,302 chunks | 95,098 tok/s | elapsed 